# Food Long-Tail OOD Challenge — Vanilla ResNet Baseline

Plainest possible baseline:
1. ResNet-50 ImageNet pretrained, fine-tune on the full training set (no long-tail tricks).
2. Predict the argmax over 101 known classes for every test image.
3. Write `submission.csv`.

Expected behavior: the model can never predict the `unknown` class (101), so the ~25% OOD test samples will all be wrong — capping the achievable Top-1 accuracy at roughly 75%. This baseline establishes the floor; better submissions add long-tail and OOD handling on top.

In [4]:
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

# =====================================================================
# 1. 基础参数与路径配置
# =====================================================================
print("⚙️ 初始化参数...")

# 沿用你验证过绝对正确的根目录
DATA_ROOT = Path('/content/Food-Classification')
OUT_SUB = DATA_ROOT / 'submission_swin_hybrid.csv'

NUM_KNOWN = 101
NUM_CLASSES = NUM_KNOWN
OOD_LABEL = 101
OOD_RATIO = 0.25        # 官方理论值 25%

IMG_SIZE = 224
BATCH_SIZE = 64         # 如果趋动云显存够大，这里可以果断改成 128
NUM_WORKERS = 4
EPOCHS = 5
LR = 2e-4               # Swin 专属微调学习率
SEED = 42

DEVICE = torch.device('cuda' if torch.cuda.is_available() else'cpu')
print(f'✅ 运算设备: {DEVICE}')

torch.manual_seed(SEED)
np.random.seed(SEED)

# =====================================================================
# 2. 原版 Dataset 数据流水线 (极简无冗余)
# =====================================================================
IMNET_MEAN = [0.485, 0.456, 0.406]
IMNET_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMNET_MEAN, IMNET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMNET_MEAN, IMNET_STD),
])

class FoodDataset(Dataset):
    """原版精简读取逻辑，无智能寻路开销"""
    def __init__(self, df, img_dir, transform, has_label):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.has_label = has_label

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.img_dir / f"{row['image_id']}.jpg").convert('RGB')
        img = self.transform(img)
        if self.has_label:
            return img, int(row['label'])
        return img, row['image_id']

train_df = pd.read_csv(DATA_ROOT / 'train.csv')
test_df = pd.read_csv(DATA_ROOT / 'test.csv')

# 沿用你 baseline(6.48) 中已被验证的套娃路径！
tr_ds = FoodDataset(train_df, DATA_ROOT / 'train_images' / 'train_images', train_tf, has_label=True)
test_ds = FoodDataset(test_df, DATA_ROOT / 'test_images' / 'test_images', eval_tf, has_label=False)

tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE*2, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
print('✅ 数据流水线准备完毕！')

# =====================================================================
# 3. 构建 Swin-Tiny 模型与温和类别平衡
# =====================================================================
print("🦁 加载现代网络 Swin-Tiny...")
model = models.swin_t(weights=models.Swin_T_Weights.IMAGENET1K_V1)
model.head = nn.Linear(model.head.in_features, NUM_KNOWN)
model = model.to(DEVICE)

# 温和版长尾平衡权重 (平方根平滑)
class_counts = train_df['label'].value_counts().sort_index().values
weights = 1.0 / (np.sqrt(class_counts) + 1.0)
weights = weights / np.sum(weights) * NUM_KNOWN
class_weights = torch.FloatTensor(weights).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# =====================================================================
# 4. 正式训练模型
# =====================================================================
print("🔥 开始 5 轮高阶微调...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    total, correct, loss_sum = 0, 0, 0.0
    bar = tqdm(tr_loader, desc=f'Swin Epoch {epoch}/{EPOCHS}', leave=False)

    for x, y in bar:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)
        bar.set_postfix(loss=loss_sum/total, acc=correct/total)

    scheduler.step()
    print(f'Epoch {epoch}/{EPOCHS} | Loss {loss_sum/total:.3f} | Train Acc {correct/total:.3f} | {time.time()-t0:.1f}s')

# =====================================================================
# 5. 特征提取与中心计算 (修复为抗维度的均值中心)
# =====================================================================
def extract_swin_features(model, x):
    """提取 Swin 的 768 维特征"""
    x = model.features(x)
    x = x.permute(0, 3, 1, 2)
    x = model.avgpool(x)
    x = torch.flatten(x, 1)
    return x

print("\n🔍 提取训练集特征并计算均值中心...")
model.eval()
features_per_class = [[] for _ in range(NUM_CLASSES)]

with torch.no_grad():
    for x, y in tqdm(tr_loader, desc="Train Features"):
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        feats = F.normalize(extract_swin_features(model, x), dim=1)
        for i in range(x.size(0)):
            features_per_class[y[i].item()].append(feats[i].cpu().numpy())

class_centers = []
for c in range(NUM_CLASSES):
    if len(features_per_class[c]) == 0:
        class_centers.append(np.zeros(768, dtype=np.float32))
    else:
        # ✅ 修复：高维空间必须使用均值
        class_centers.append(np.stack(features_per_class[c], axis=0).mean(axis=0))

class_centers = F.normalize(torch.tensor(np.stack(class_centers), dtype=torch.float32).to(DEVICE), dim=1)

# =====================================================================
# 6. 全局双轨混合动力预测与导出
# =====================================================================
print("\n🚀 正在收集测试集全局特征与置信度...")
all_ids, all_preds_sim, all_max_sims, all_max_logits = [], [], [], []

with torch.no_grad():
    for x, ids in tqdm(test_loader, desc="Testing"):
        x = x.to(DEVICE, non_blocking=True)

        logits = model(x)
        feats = F.normalize(extract_swin_features(model, x), dim=1)

        # 特征相似度
        sims = feats @ class_centers.T
        max_sim, pred_sim = sims.max(dim=1)

        # 原始置信度
        max_logit, _ = logits.max(dim=1)

        all_ids.extend(ids)
        all_preds_sim.extend(pred_sim.cpu().numpy().tolist())
        all_max_sims.extend(max_sim.cpu().numpy().tolist())
        all_max_logits.extend(max_logit.cpu().numpy().tolist())

print("🧠 全局融合计算与 25% OOD 拦截...")
all_max_sims = np.array(all_max_sims)
all_max_logits = np.array(all_max_logits)

# 真正的全局归一化
norm_logits = (all_max_logits - all_max_logits.mean()) / (all_max_logits.std() + 1e-5)
score_logits = 1.0 / (1.0 + np.exp(-norm_logits))

# 数学双剑合璧
all_hybrid_scores = 0.7 * all_max_sims + 0.3 * score_logits

threshold = np.quantile(all_hybrid_scores, OOD_RATIO)
final_preds = [OOD_LABEL if score <= threshold else pred for pred, score in zip(all_preds_sim, all_hybrid_scores)]

print(f"📊 判决完毕：成功拦截 {final_preds.count(OOD_LABEL)} 个 OOD 样本！")

# 导出提交
sub = pd.DataFrame({'image_id': all_ids, 'label': final_preds})
order = pd.read_csv(DATA_ROOT / 'test.csv')['image_id'].tolist()
sub = sub.set_index('image_id').loc[order].reset_index()
sub.to_csv(OUT_SUB, index=False)
print(f'✅ 终极版文件已生成！路径: {OUT_SUB}')


⚙️ 初始化参数...
✅ 运算设备: cuda
✅ 数据流水线准备完毕！
🦁 加载现代网络 Swin-Tiny...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Downloading: "https://download.pytorch.org/models/swin_t-704ceda3.pth" to /root/.cache/torch/hub/checkpoints/swin_t-704ceda3.pth


100%|██████████| 108M/108M [00:00<00:00, 223MB/s] 


🔥 开始 5 轮高阶微调...


...

Train Features:   0%|          | 0/514 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b12830c2e80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process



🚀 正在收集测试集全局特征与置信度...


Testing:   0%|          | 0/158 [00:00<?, ?it/s]

🧠 全局融合计算与 25% OOD 拦截...
📊 判决完毕：成功拦截 5045 个 OOD 样本！
✅ 终极版文件已生成！路径: /content/Food-Classification/submission_swin_hybrid.csv
